**DSPy GRPO Statistics Notebook** — experiment logging for GRPO runs (accuracy, inference time, K sweeps, episode sweeps). Baseline algorithm matches `DSPy_GRPO.ipynb`.


**Installing Dependencies**

In [ ]:
!pip install -U dspy
!pip install datasets
!pip install torch transformers

**Setting Up Ollama**

In [ ]:
# Windows: Ollama is installed separately - skip Linux install script.
# !curl -fsSL https://ollama.com/install.sh | sh
print("Skipping Ollama Linux install (using existing Windows Ollama).")


In [ ]:
# Ollama is already running externally - do not start another server.
# import subprocess
# process = subprocess.Popen("ollama serve", shell=True)
print("Skipping 'ollama serve' (Ollama already running at http://localhost:11434).")


In [ ]:
# llama3.1:8b is already downloaded locally.
# !ollama pull llama3.1:8b
print("Skipping 'ollama pull llama3.1:8b' (model already present).")


**Import Libraries**

In [ ]:
import torch
from transformers import GPT2Tokenizer, GPT2LMHeadModel
import dspy
from datasets import load_dataset
import numpy as np
import re
import matplotlib.pyplot as plt
import time
import random
import json
from pprint import pprint
import logging
# Ollama already running externally - no subprocess server start.


**Configure DSPy & LLM**

In [ ]:
# === Configure DSPy with Ollama LLM ===
lm = dspy.LM('ollama_chat/llama3.2:3b', api_base='http://127.0.0.1:11434/v1', api_key='ollama')
dspy.configure(lm=lm)

**GRPO Experiment Configuration**


In [ ]:
# === Single place to control GRPO statistics experiments ===
EXPERIMENT_CONFIG = {
    # Reproducibility
    "random_seed": 42,
    # Training data: "gsm8k", "hotpotqa", or "both" (combined shuffle)
    "train_dataset": "gsm8k",
    # Core training run (used when run_training=True)
    "experiment_name": "grpo_smoke_log",
    "policy_model_name": "gpt2",
    "K": 4,
    "num_episodes": 2,
    "num_training_samples_per_dataset": 10,
    "learning_rate": 2e-5,
    "checkpoint_dir": "checkpoints",
    "auto_checkpoint_path": True,
    "save_path": None,
    # Evaluation (used when run_eval=True)
    "eval_sample_count": 30,
    "eval_dataset": "both",
    "eval_start_index": 0,
    # GSM8K uses test[:eval_sample_count]. HotPotQA uses train[20:20+eval_sample_count].
    # Planned sweeps (enable flags below; keep False for short runs)
    "K_values_to_compare": [5, 10, 15],
    "episode_counts_to_compare": [10, 50, 100],
    # Persistence
    "results_json": "policy_model_comparison_results.json",
    "results_csv": "policy_model_comparison_results.csv",
    # Execution switches
    "run_training": True,
    "run_eval": False,
    "run_k_sweep": False,
    "run_episode_sweep": False,
}

# Derived after data load
EXPERIMENT_CONFIG["num_training_samples"] = None


**Load Training Datasets**

In [ ]:
# === Load GSM8K and HotpotQA for training/testing ===
print("Loading datasets...")
_n = EXPERIMENT_CONFIG["num_training_samples_per_dataset"]
gsm8k = load_dataset("openai/gsm8k", "main", split=f"train[:{_n}]")
hotpotqa = load_dataset("hotpotqa/hotpot_qa", "fullwiki", split=f"train[:{_n}]")

gsm8k_list = list(gsm8k)
hotpotqa_list = list(hotpotqa)

prompts = []
ground_truths = []


def load_training_data_for_experiment(config):
    """Select GSM8K, HotPotQA, or combined training pool for one isolated run."""
    global prompts, ground_truths

    train_dataset = config.get("train_dataset", "gsm8k")
    gsm8k_data = [
        (ex["question"], ex["answer"].split("####")[-1].strip()) for ex in gsm8k_list
    ]
    hotpotqa_data = [(ex["question"], ex["answer"]) for ex in hotpotqa_list]

    if train_dataset == "gsm8k":
        data = gsm8k_data
    elif train_dataset == "hotpotqa":
        data = hotpotqa_data
    elif train_dataset == "both":
        data = gsm8k_data + hotpotqa_data
        rng = random.Random(config.get("random_seed", 42))
        rng.shuffle(data)
    else:
        raise ValueError(f"Unknown train_dataset: {train_dataset}")

    prompts = [x for x, _ in data]
    ground_truths = [y for _, y in data]
    config["num_training_samples"] = len(prompts)
    print(
        f"Training data prepared: dataset={train_dataset}, "
        f"samples={config['num_training_samples']}"
    )


load_training_data_for_experiment(EXPERIMENT_CONFIG)


**Action Space & Pool**

In [ ]:
# === Define Action Space ===
MODULES = ["CoT", "Predict"]
SIGNATURES = [
    "question -> answer", "text -> summary", "question -> reasoning", "question -> hypothesis",
    "problem -> solution", "problem_description -> explanation", "context -> summary", "context -> briefing",
    "word_problem -> solution", "math_problem -> answer", "prompt -> response", "query -> response",
    "text -> response", "prompt -> generated_text", "query -> generated_text"
]
ACTIONS = MODULES + SIGNATURES + ["stop"]

**Policy Model Setup**

In [ ]:
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

policy_device = torch.device("cpu")


def create_fresh_policy_model():
    """Load a new pretrained GPT-2 policy network (no checkpoint reuse)."""
    model = GPT2LMHeadModel.from_pretrained("gpt2")
    model.config.pad_token_id = tokenizer.pad_token_id
    model.to(policy_device)
    return model


policy_model = create_fresh_policy_model()
print(f"Using device: {policy_device}")
print("GPT-2 policy model initialized from pretrained weights (no checkpoint load)")


**Helper Functions**

In [ ]:
# ============================================================
# Compare Answers Functions
def compare_answers(response, ground_truth):
    """
    Extracts the final answer from both the response and the ground truth,
    then compares them. The extraction first looks for bracketed content;
    if not found, it falls back to the last numeric match.

    Returns:
        regex_score (float): 1.0 if the answers match, 0.0 otherwise.
    """
    # Extract the answer from response: first try bracketed content
    response_bracket_matches = re.findall(r'\[(.*?)\]', str(response))
    if response_bracket_matches:
        response_answer = response_bracket_matches[-1].strip()
        print(f"Response bracket match: {response_answer}")
    else:
        response_num_matches = re.findall(r'-?\d*\.?\d+', str(response))
        response_answer = response_num_matches[-1] if response_num_matches else None
        print(f"Response last numeric match: {response_answer}")

    # Extract answer from ground truth similarly
    gt_bracket_matches = re.findall(r'\[(.*?)\]', str(ground_truth))
    if gt_bracket_matches:
        gt_answer = gt_bracket_matches[-1].strip()
        print(f"Ground truth bracket match: {gt_answer}")
    else:
        gt_num_matches = re.findall(r'-?\d*\.?\d+', str(ground_truth))
        gt_answer = gt_num_matches[-1] if gt_num_matches else None
        print(f"Ground truth last numeric match: {gt_answer}")

    # Compare the extracted answers
    if response_answer and gt_answer:
        response_num_match = re.search(r'-?\d*\.?\d+', response_answer)
        gt_num_match = re.search(r'-?\d*\.?\d+', gt_answer)
        if response_num_match and gt_num_match:
            response_num = response_num_match.group(0)
            gt_num = gt_num_match.group(0)
            regex_score = 1.0 if response_num == gt_num else 0.0
            print(f"Numeric comparison - Response: {response_num}, Ground Truth: {gt_num}, Regex Score: {regex_score}")
        else:
            regex_score = 1.0 if response_answer == gt_answer else 0.0
            print(f"String comparison - Response: {response_answer}, Ground Truth: {gt_answer}, Regex Score: {regex_score}")
    else:
        regex_score = 0.0
        print("No valid match found, Regex Score: 0.0")

    return regex_score

# ============================================================
# Compute Reward Function
def compute_reward(prompt, response, ground_truth):
    """
    Computes a reward for the response.
    First, a DSPy-format bonus is given if the response strictly matches the desired format.
    Then, it attempts to extract the final answer using regex (preferring bracketed content).
    The reward is based on matching the numeric (or string) content with the ground truth.
    """
    if response is None:
        print("Response is None")
        return 0.0

    regex1 = compare_answers(response, ground_truth)
    score = regex1
    print("score: ", score)

    # If regex score1 is 0, fall back to LLM judge
    if regex1 == 0.0:
        eval_prompt = f"""
        Evaluate whether the following response correctly answers the prompt based on the ground truth and give your final score in the square brackets[final score]. only the value in the [] should in your response and nothing else.
        Prompt: {prompt}
        Response: {response}
        Ground Truth: {ground_truth}
        Return a score in range between 1.0 to 0.0 if the response is correct or partially correct (matches or is equivalent to the ground truth), or 0.0 if incorrect.
        final score:[]
            """
        try:
            llm_response = lm(eval_prompt)
            print("LLM Response: ", llm_response)
            response_bracket_matches = re.findall(r'\[(.*?)\]', (llm_response[0]))
            print("llm matches: ", response_bracket_matches)
            if response_bracket_matches:
                response_answer = response_bracket_matches[-1].strip()
                print(f"llm Response bracket match: {response_answer}")
            else:
                response_num_matches = re.findall(r'-?\d*\.?\d+', (llm_response[0]))
                response_answer = response_num_matches[-1] if response_num_matches else None
                print(f"llm Response last numeric match: {response_answer}")
            score_str = response_answer
            score = float(score_str.strip())
            print(f"LLM score: {score_str}, in float: {score}")
            return min(max(score, 0.0), 1.0)

        except (ValueError, TypeError, IndexError):
            print("LLM evaluation failed, defaulting to 0.0")
            return 0.0

    return score

# ============================================================
# Generate Pipeline Function
def generate_pipeline(prompt, max_steps=3):
    """
    Generate a discrete DSPy pipeline using a fixed set of actions.
    This function restricts the output to a sequence of actions chosen from MODULES, SIGNATURES, and "stop".
    Note: Each action is approximated by using the logit of its first token.
    """
    state = {"prompt": prompt, "partial_pipeline": []}
    actions_taken = []
    log_probs = []

    for _ in range(max_steps):
        input_text = f"Prompt: {state['prompt']} Pipeline: {' '.join(state['partial_pipeline'])}"
        inputs = tokenizer(input_text, return_tensors="pt", padding=True, truncation=True).to(policy_device)
        outputs = policy_model(**inputs)
        logits = outputs.logits[:, -1, :]

        if not state["partial_pipeline"]:
            valid_actions = MODULES
        elif len(state["partial_pipeline"]) == 1 and state["partial_pipeline"][0] in MODULES:
            valid_actions = SIGNATURES
        elif len(state["partial_pipeline"]) == 2:
            valid_actions = ["stop"]
        else:
            break

        # Map each valid action to a token id (using first token as approximation)
        valid_token_ids = []
        for a in valid_actions:
            tokens = tokenizer.encode(a, add_special_tokens=False)
            if len(tokens) == 0:
                continue
            valid_token_ids.append(tokens[0])

        # Get logits and sample one action
        action_logits = logits[0, valid_token_ids]
        action_probs = torch.softmax(action_logits, dim=-1)
        action_choice = torch.multinomial(action_probs, 1).item()
        chosen_token_id = valid_token_ids[action_choice]
        action = valid_actions[action_choice]
        log_prob = torch.log(action_probs[action_choice])

        actions_taken.append(action)
        log_probs.append(log_prob)
        state["partial_pipeline"].append(action)

        if action == "stop":
            break

    return state["partial_pipeline"], actions_taken, log_probs

# ============================================================
# Execute Pipeline Function
def execute_pipeline(prompt, pipeline):
    """
    Execute the DSPy pipeline using the dspy modules.
    The pipeline should be a list of two actions: [module, signature] (with an optional "stop" appended).
    """
    if len(pipeline) != 2 or pipeline[0] not in MODULES or pipeline[1] not in SIGNATURES:
        print("Invalid pipeline format")
        return None

    module, signature = pipeline

    if signature.count("->") != 1:
        print("Signature does not have proper format")
        return None

    match = re.match(r"\s*([a-zA-Z_ ]+)\s*->\s*([a-zA-Z_ ]+)\s*", signature)
    if not match:
        print("Regex match failed for signature")
        return None

    inputfield = match.group(1).strip().replace(" ", "_").lower()
    outputfield = match.group(2).strip().replace(" ", "_").lower()

    try:
        if module == "CoT":
            program = dspy.ChainOfThought(signature)
        else:  # "Predict"
            program = dspy.Predict(signature)
        response = program(**{inputfield: f"system instruction: Must give your final answer in square brackets without fail! e.g., [final answer] like this: [5] \n prompt: {prompt}"})
        return response.get(outputfield)
    except Exception as e:
        print(f"Execution pipeline failed: {e}")
        return None

# ============================================================
# Testing The Trained Model Function
def test_model(trained_model, test_prompt, max_steps=3):
    """
    Evaluate the trained model on a test prompt by generating a discrete DSPy pipeline
    and then executing it.
    """
    trained_model.eval()
    with torch.no_grad():
        pipeline, actions, _ = generate_pipeline(test_prompt, max_steps=max_steps)
        response = execute_pipeline(test_prompt, pipeline[:-1] if pipeline[-1] == "stop" else pipeline)
        print(f"Test Prompt: {test_prompt}")
        print(f"Generated Pipeline: {pipeline}")
        print(f"Actions Taken: {actions}")
        print(f"Response: {response}")
    return response

In [ ]:
# === GRPO experiment logging (does not change GRPO update logic) ===
import csv
from pathlib import Path
from datetime import datetime, timezone


def _utc_now():
    return datetime.now(timezone.utc).isoformat()


def set_experiment_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def reset_policy_model_from_pretrained():
    """Ensure each experiment starts from fresh pretrained GPT-2 weights."""
    global policy_model
    policy_model = create_fresh_policy_model()
    policy_model.train()
    print("Reset policy_model from pretrained GPT-2")


def build_checkpoint_path(config):
    dataset = config.get("train_dataset", "gsm8k")
    K = config.get("K")
    num_episodes = config.get("num_episodes")
    exp_name = config.get("experiment_name", "grpo")
    checkpoint_dir = Path(config.get("checkpoint_dir", "checkpoints"))
    checkpoint_dir.mkdir(parents=True, exist_ok=True)
    filename = f"{exp_name}_{dataset}_K{K}_ep{num_episodes}.pt"
    return str(checkpoint_dir / filename)


def resolve_experiment_config(config=None):
    config = dict(config or EXPERIMENT_CONFIG)
    if config.get("auto_checkpoint_path", True) or not config.get("save_path"):
        config["save_path"] = build_checkpoint_path(config)
    return config


def save_experiment_record(record: dict) -> dict:
    """Append one experiment row to JSON and CSV."""
    row = {k: record.get(k) for k in [
        "timestamp",
        "experiment_name",
        "phase",
        "policy_model_name",
        "train_dataset",
        "random_seed",
        "num_episodes",
        "K",
        "average_reward",
        "final_episode_reward",
        "dead_episode_count",
        "dead_episode_rate",
        "training_time_seconds",
        "checkpoint_path",
        "dataset",
        "eval_sample_count",
        "accuracy",
        "inference_time_per_example",
        "inference_time_total",
        "reward_variance",
        "num_training_samples",
        "notes",
        "eval_start_index",
        "examples_evaluated",
    ]}
    row["timestamp"] = row.get("timestamp") or _utc_now()

    json_path = EXPERIMENT_CONFIG["results_json"]
    if Path(json_path).exists():
        existing = json.loads(Path(json_path).read_text(encoding="utf-8"))
        if not isinstance(existing, list):
            existing = [existing]
    else:
        existing = []
    existing.append(row)
    Path(json_path).write_text(json.dumps(existing, indent=2), encoding="utf-8")

    csv_path = EXPERIMENT_CONFIG["results_csv"]
    csv_file = Path(csv_path)
    if csv_file.exists():
        existing_text = csv_file.read_text(encoding="utf-8")
        existing_lines = existing_text.splitlines()
        if existing_lines:
            header_names = existing_lines[0].split(",")
            missing = [key for key in row if key not in header_names]
            if missing:
                existing_lines[0] = existing_lines[0] + "," + ",".join(missing)
                csv_file.write_text("\n".join(existing_lines) + "\n", encoding="utf-8")
    write_header = not csv_file.exists()
    with open(csv_path, "a", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=list(row.keys()))
        if write_header:
            writer.writeheader()
        writer.writerow(row)

    print("Logged experiment record:", row)
    return row


def train_grpo_with_metrics(num_episodes=500, learning_rate=1e-4, K=4, save_path="gpt2_trained_policy_model_grpo.pt"):
    """Wrapper around train_grpo; returns model + reward statistics."""
    model, metrics = train_grpo(
        num_episodes=num_episodes,
        learning_rate=learning_rate,
        K=K,
        save_path=save_path,
    )
    return model, metrics


def evaluate_dataset_logged(dataset_name, dataset, model, query=False, output_file=None, config=None):
    """Run evaluate_module and persist accuracy + inference timing."""
    config = config or EXPERIMENT_CONFIG
    if output_file is None:
        output_file = f"evaluation_{dataset_name}.json"
    total_time, total_correct = evaluate_module(
        dataset, None, model, query=query, output_file=output_file
    )
    n = len(dataset)
    start = int(config.get("eval_start_index") or 0)
    accuracy = (total_correct / n) if n else 0.0
    avg_time = (total_time / n) if n else 0.0
    record = {
        "experiment_name": config.get("experiment_name"),
        "phase": "eval",
        "policy_model_name": config.get("policy_model_name"),
        "train_dataset": config.get("train_dataset"),
        "random_seed": config.get("random_seed"),
        "K": config.get("K"),
        "num_episodes": config.get("num_episodes"),
        "num_training_samples": config.get("num_training_samples"),
        "dataset": dataset_name,
        "eval_sample_count": int(config.get("eval_sample_count", n)),
        "eval_start_index": start,
        "examples_evaluated": int(n),
        "accuracy": float(accuracy),
        "inference_time_per_example": float(avg_time),
        "inference_time_total": float(total_time),
        "checkpoint_path": config.get("save_path"),
        "notes": f"eval_n={n}, eval_start_index={start}, query={query}",
    }
    save_experiment_record(record)
    return accuracy, avg_time, total_time


def run_single_grpo_experiment(config=None):
    config = resolve_experiment_config(config)
    if not config.get("run_training"):
        print("run_training=False; skipping training.")
        return None, None, config

    set_experiment_seed(config["random_seed"])
    load_training_data_for_experiment(config)
    reset_policy_model_from_pretrained()

    print(f"Starting isolated training run: {config['save_path']}")
    model, metrics = train_grpo_with_metrics(
        num_episodes=config["num_episodes"],
        learning_rate=config["learning_rate"],
        K=config["K"],
        save_path=config["save_path"],
    )

    save_experiment_record({
        "experiment_name": config.get("experiment_name"),
        "phase": "training",
        "policy_model_name": config.get("policy_model_name"),
        "train_dataset": config.get("train_dataset"),
        "random_seed": config.get("random_seed"),
        "num_episodes": config.get("num_episodes"),
        "K": config.get("K"),
        "num_training_samples": config.get("num_training_samples"),
        "average_reward": metrics.get("average_reward"),
        "final_episode_reward": metrics.get("final_episode_reward"),
        "dead_episode_count": metrics.get("dead_episode_count"),
        "dead_episode_rate": metrics.get("dead_episode_rate"),
        "training_time_seconds": metrics.get("training_time_seconds"),
        "reward_variance": metrics.get("reward_variance"),
        "checkpoint_path": config.get("save_path"),
        "notes": "post_train_metrics",
    })
    return model, metrics, config


def run_k_sweep(K_values=None, base_config=None):
    K_values = K_values or EXPERIMENT_CONFIG["K_values_to_compare"]
    base_config = dict(base_config or EXPERIMENT_CONFIG)
    results = []
    for K in K_values:
        cfg = resolve_experiment_config({
            **base_config,
            "K": K,
            "run_training": True,
            "experiment_name": f"{base_config['experiment_name']}_K{K}",
        })
        print(f"=== Isolated K sweep run: K={K} ===")
        model, metrics, cfg = run_single_grpo_experiment(cfg)
        results.append({"K": K, "metrics": metrics, "save_path": cfg["save_path"]})
    return results


def run_episode_sweep(episode_counts=None, base_config=None):
    episode_counts = episode_counts or EXPERIMENT_CONFIG["episode_counts_to_compare"]
    base_config = dict(base_config or EXPERIMENT_CONFIG)
    results = []
    for num_episodes in episode_counts:
        cfg = resolve_experiment_config({
            **base_config,
            "num_episodes": num_episodes,
            "run_training": True,
            "experiment_name": f"{base_config['experiment_name']}_ep{num_episodes}",
        })
        print(f"=== Isolated episode sweep run: num_episodes={num_episodes} ===")
        model, metrics, cfg = run_single_grpo_experiment(cfg)
        results.append({
            "num_episodes": num_episodes,
            "metrics": metrics,
            "save_path": cfg["save_path"],
        })
    return results


**Training Model**

In [ ]:
def train_grpo(num_episodes=500, learning_rate=1e-4, K=4, save_path="gpt2_trained_policy_model_grpo.pt"):
    """
    Train the policy model using a GRPO-inspired update with group-based advantage estimation.
    For each episode, generate K pipelines per prompt, compute group average reward, and use
    reward - group_average as the advantage for policy updates.
    """
    optimizer = torch.optim.Adam(policy_model.parameters(), lr=learning_rate)
    rewards = []
    losses = []
    dead_episode_count = 0
    training_loop_start = time.perf_counter()

    for episode in range(num_episodes):
        idx = np.random.randint(len(prompts))
        prompt = prompts[idx]
        ground_truth = ground_truths[idx]

        # Generate K pipelines for the same prompt
        pipelines = []
        actions_list = []
        log_probs_list = []
        rewards_list = []

        for _ in range(K):
            pipeline, actions, log_probs = generate_pipeline(prompt)
            # Execute discrete DSPy module (exclude "stop" if present)
            response = execute_pipeline(prompt, pipeline[:-1] if pipeline[-1] == "stop" else pipeline)
            reward = compute_reward(prompt, response, ground_truth)
            pipelines.append(pipeline)
            actions_list.append(actions)
            log_probs_list.append(log_probs)
            rewards_list.append(reward)

        # Compute group average reward
        group_average = np.mean(rewards_list)

        # Compute advantages for each pipeline
        advantages = [reward - group_average for reward in rewards_list]
        if advantages and all(advantage == 0 for advantage in advantages):
            dead_episode_count += 1

        # Compute policy loss
        loss = 0
        for k in range(K):
            advantage = advantages[k]
            log_probs = log_probs_list[k]
            for log_prob in log_probs:
                loss -= log_prob * advantage

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # Log average reward and loss
        avg_reward = np.mean(rewards_list)
        rewards.append(avg_reward)
        losses.append(loss.item())

        print(f"Episode {episode}, Prompt: {prompt[:20]}..., Avg Reward: {avg_reward:.3f}, Loss: {loss.item():.3f}")

    training_time_seconds = time.perf_counter() - training_loop_start

    torch.save(policy_model.state_dict(), save_path)
    print(f"Model saved to {save_path}")
    episode_rewards = [float(r) for r in rewards]
    training_metrics = {
        "average_reward": float(np.mean(rewards)) if rewards else 0.0,
        "reward_variance": float(np.var(rewards)) if rewards else 0.0,
        "episode_rewards": episode_rewards,
        "final_episode_reward": float(episode_rewards[-1]) if episode_rewards else 0.0,
        "dead_episode_count": int(dead_episode_count),
        "dead_episode_rate": float(dead_episode_count / num_episodes) if num_episodes else 0.0,
        "training_time_seconds": float(training_time_seconds),
    }
    return policy_model, training_metrics


**Run Training**

In [ ]:
# === Run configured GRPO experiment (short by default) ===
trained_model, training_metrics, active_config = run_single_grpo_experiment(EXPERIMENT_CONFIG)
print("Training metrics:", training_metrics)
print("Checkpoint:", active_config.get("save_path") if active_config else None)

if EXPERIMENT_CONFIG.get("run_k_sweep"):
    k_sweep_results = run_k_sweep()
    print("K sweep complete:", k_sweep_results)

if EXPERIMENT_CONFIG.get("run_episode_sweep"):
    episode_sweep_results = run_episode_sweep()
    print("Episode sweep complete:", episode_sweep_results)


In [ ]:
def evaluate_module(dataset, fulldataset, model, query=False, output_file="evaluation_results.json"):
    """
    Evaluates the given module on a dataset of examples and stores the results in a JSON file.

    Parameters:
      dataset: A list of examples, where each example has at least:
               - example.question (input text)
               - example.answer (expected answer)
      model: The model used to generate responses.
      query: If True, retrieves context from Wikipedia.
      output_file: The JSON file where results will be stored.

    Returns:
      total_time: Total elapsed time (in seconds).
      total_correct: Number of correct responses.
    """
    total_correct = 0
    results = []
    start_time = time.time()

    print("Query mode:", query)

    for i, example in enumerate(dataset):
        print(f"-------------------------------------Iteration: {i}------------------------------------------------")

        prompt = f"{example.context} \n\n {example.question} " if query else example.question
        prediction_text = test_model(model, prompt)

        if example.answer:
            is_correct = (
                prediction_text and prediction_text.strip().lower() == example.answer.strip().lower()
            ) or (
                prediction_text and example.answer.strip().lower() in prediction_text.strip().lower()
            )
        else:
            is_correct = False

        score = 1 if is_correct else 0

        if not is_correct:
            eval_prompt = f"""
                Evaluate whether the following response correctly answers the prompt based on the ground truth.
                Return a score of 1.0 or 0.0 if the response is correct and 0 if incorrect.
                Only return the score inside the square brackets [].

                Prompt: {example.question}
                Response: {prediction_text}
                Ground Truth: {example.answer}
                Final score:[]"""

            llm_res = lm(eval_prompt)
            score = 1 if "1" in llm_res[0] else 0

        total_correct += score

        print("Ground Truth: ", example.answer)
        print("Score: ", score)
        print("------------------------ total correct so far: ", total_correct, "--------------------------")
        if query:
            result = {
                "question": example.question,
                "context": example.context,
                "response": prediction_text,
                "ground_truth": example.answer if example.answer else "N/A",
                "score": score,
            }
            results.append(result)

    total_time = time.time() - start_time

    if query:
        with open(output_file, "w", encoding="utf-8") as f:
            json.dump(results, f, indent=4)

    return total_time, total_correct

**Loading Test Datasets**

In [ ]:
_eval_n = int(EXPERIMENT_CONFIG["eval_sample_count"])
_hotpot_eval_start = 20
gsm8k_test = load_dataset("openai/gsm8k", "main", split=f"test[:{_eval_n}]")
hotpotqa_test = load_dataset(
    "hotpotqa/hotpot_qa",
    "fullwiki",
    split=f"train[{_hotpot_eval_start}:{_hotpot_eval_start + _eval_n}]",
)

gsm8k_test_data = [dspy.Example(question=ex['question'], answer=ex["answer"].split("####")[-1].strip(), task_type='math').with_inputs('question') for ex in gsm8k_test]
hotpotqa_test_data = [dspy.Example(question=ex['question'], answer=ex['answer'], context=ex["context"], task_type='qa').with_inputs('question') for ex in hotpotqa_test]
print(f"Eval sets loaded: GSM8K={len(gsm8k_test_data)}, HotPotQA={len(hotpotqa_test_data)}")


In [ ]:
# Eval HotPotQA (logged) — set EXPERIMENT_CONFIG["run_eval"] = True
_eval_dataset = EXPERIMENT_CONFIG.get("eval_dataset", "both")
_eval_start = int(EXPERIMENT_CONFIG.get("eval_start_index") or 0)
_eval_end = int(EXPERIMENT_CONFIG["eval_sample_count"])
if EXPERIMENT_CONFIG.get("run_eval") and _eval_dataset in ("hotpotqa", "both"):
    hotpotqa_eval_slice = hotpotqa_test_data[_eval_start:_eval_end]
    h_accuracy, h_avg_time, h_total_time = evaluate_dataset_logged(
        "hotpotqa", hotpotqa_eval_slice, policy_model, query=True
    )
    print(
        f"HotPotQA - accuracy: {h_accuracy:.3f}, "
        f"examples: {len(hotpotqa_eval_slice)} from index {_eval_start}, "
        f"avg time/example: {h_avg_time:.2f}s, total: {h_total_time:.2f}s"
    )
else:
    print("Skipping HotPotQA eval.")


In [ ]:
# Eval GSM8K (logged) — set EXPERIMENT_CONFIG["run_eval"] = True
_eval_dataset = EXPERIMENT_CONFIG.get("eval_dataset", "both")
_eval_start = int(EXPERIMENT_CONFIG.get("eval_start_index") or 0)
_eval_end = int(EXPERIMENT_CONFIG["eval_sample_count"])
if EXPERIMENT_CONFIG.get("run_eval") and _eval_dataset in ("gsm8k", "both"):
    gsm8k_eval_slice = gsm8k_test_data[_eval_start:_eval_end]
    g_accuracy, g_avg_time, g_total_time = evaluate_dataset_logged(
        "gsm8k", gsm8k_eval_slice, policy_model, query=False
    )
    print(
        f"GSM8K - accuracy: {g_accuracy:.3f}, "
        f"examples: {len(gsm8k_eval_slice)} from index {_eval_start}, "
        f"avg time/example: {g_avg_time:.2f}s, total: {g_total_time:.2f}s"
    )
else:
    print("Skipping GSM8K eval.")
